# Lab 4：MLP、编码器与跨模态 latent 转换

目标：运行完整的 `仿真几何 → 编码器 → latent 映射 → 传感器解码器 → 三轴输出`。

本 notebook 默认使用五条**实测记录**。可以切换到由相同轨迹派生、包含切向偏移和速度变化的 **synthetic.npz**；合成信号不是新采集的真机数据，合成成绩也不代表硬件精度。

课堂不需要采集新数据或调场景。先观看仓库中的 `lab4/figures/replay_comparison.gif`：左边为原始回放，右边为带固定虚拟工装偏移的合成几何回放。右侧传感器曲线由教学模型产生，未声称复现了真实接触力。

建议：先运行全部单元得到基线，再做末尾练习。完整训练通常只需 CPU；时间以你的机器为准。

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'lab4').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'lab4').is_dir(), 'Start the notebook from the repository root or lab4 directory'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab4.data import load_windows
from lab4.models import mlp, Autoencoder, LatentTranslator
from lab4.train import fit, metrics

DATA_DIR = ROOT / 'lab4/datasets'
DOMAIN = 'measured'             # Switch to synthetic for the generated dataset
WINDOW = 7                     # Current and six past packets; synthetic duration depends on speed
EPOCHS = 600
SIM_LATENT = 8
SENSOR_LATENT = 4
SEED = 7

torch.set_num_threads(2)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
data = load_windows(DATA_DIR / f'{DOMAIN}.npz', window=WINDOW)
print('Target source:', data['source_kind'])
for split in ['train', 'validation', 'test']:
    m = data['split'] == split
    print(split, 'windows=', m.sum(), 'parents=', np.unique(data['parent'][m]))


## 1. 一对样本是什么

每个时刻，仿真提供 11 维几何特征：局部最近点、局部法向、有符号距离、距离变化速度、几何压入量和接触指示和包间隔。最近点不一定是实际接触点。

目标是 ID3 的三轴传感器输出。7 个时刻组成一个因果窗口，因此输入为 77 维、目标历史为 21 维。实测数据经过 SDK 名义单位换算；没有把仿真力复制成真机力。

同一原始 episode 派生的全部序列属于同一个 split。标准化只在训练集拟合。

In [ ]:
x = torch.from_numpy(data['x'])
y = torch.from_numpy(data['y'])
train = data['split'] == 'train'
validation = data['split'] == 'validation'
test = data['split'] == 'test'
xt, xv = x[train], x[validation]
yt, yv = y[train], y[validation]
print('All input windows:', tuple(x.shape), 'All target windows:', tuple(y.shape))
print('Features:', data['feature_names'].tolist())

seq = np.unique(data['sequence'][train])[0]
selected = data['sequence'] == seq
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(data['time'][selected], data['target'][selected])
ax.legend(['x', 'y', 'z']); ax.set(xlabel='Time (s)', ylabel='Nominal N', title=seq)
ax.grid(alpha=.2)
plt.show()


## 2. 直接 MLP：先不做 latent 转换

MLP 用若干线性层和非线性激活，将 77 维窗口映射为当前的 3 维输出。训练目标是标准化后的均方误差。

我们同时创建后续网络，保证固定种子的参考实验可复现。`fit` 只用验证集选 epoch，测试集不参与训练和选模型。

In [ ]:
direct = mlp(x.shape[1], 3)
sim_ae = Autoencoder(x.shape[1], SIM_LATENT)
sensor_ae = Autoencoder(y.shape[1], SENSOR_LATENT)
print(direct)
direct_history = fit(direct, xt, yt[:, -3:], xv, yv[:, -3:], epochs=EPOCHS)
print('MLP selected epoch:', direct_history['best_epoch'])


## 3. 两个自编码器：每个模态先重建自己

仿真：`77 → encoder → 8 → decoder → 77`。

传感器：`21 → encoder → 4 → decoder → 21`。

瓶颈向量叫 latent。它是任务学习到的表示，没有自动赋予每个维度接触位置或压力的物理意义。两个独立训练的 latent 坐标也不天然相同。

In [ ]:
sim_history = fit(sim_ae, xt, xt, xv, xv, epochs=EPOCHS)
sensor_history = fit(sensor_ae, yt, yt, yv, yv, epochs=EPOCHS)
with torch.no_grad():
    z_sim_example = sim_ae.encoder(xt[:5])
    z_sensor_example = sensor_ae.encoder(yt[:5])
print('Simulation latent:', tuple(z_sim_example.shape))
print('Sensor latent:', tuple(z_sensor_example.shape))
print('Sensor reconstruction validation loss:', sensor_history['best_validation_loss'])


## 4. 编码转换：冻结两端，只学习中间的映射

训练时：`z_sim = E_sim(x_sim)`，`z_sensor = E_sensor(x_sensor)`，学习 `T(z_sim) ≈ z_sensor`，并通过传感器 decoder 检查解码信号。

推理时：**只输入仿真窗口**，执行 `D_sensor(T(E_sim(x_sim)))`。不需要真实传感器信号。

冻结两端可以让学生清楚看到“中间网络到底学什么”；损失由解码重建和归一化的 latent 误差组成。

In [ ]:
translator = LatentTranslator(sim_ae, sensor_ae, SIM_LATENT, SENSOR_LATENT)
with torch.no_grad():
    translator.sensor_latent_scale.copy_(sensor_ae.encoder(yt).std(dim=0, unbiased=False).clamp_min(.1))
translation_history = fit(translator, xt, yt, xv, yv, epochs=EPOCHS,
                          objective=translator.objective)
print('Trainable parameters:', sum(p.numel() for p in translator.parameters() if p.requires_grad))
print('Latent mapping selected epoch:', translation_history['best_epoch'])


## 5. 测试：看留出的原始记录

分别比较均值基线、直接 MLP、latent 转换。传感器自编码器的自重建也列出来，但它在推理时读取了目标，只是信息压缩的参考，不能算作跨模态预测。

损失先反标准化，误差才有名义 N 单位。下面的测试只包含一个原始 episode，适合课堂对照，不能据此宣称跨物体泛化。

In [ ]:
scaler = data['y_scaler']
with torch.no_grad():
    pred_direct = scaler.inverse(direct(x).numpy())
    pred_translated = scaler.inverse(translator(x).numpy()[:, -3:])
    pred_self = scaler.inverse(sensor_ae(y).numpy()[:, -3:])
pred_mean = np.broadcast_to(data['target'][train].mean(axis=0), data['target'].shape)
for name, pred in [('Training mean', pred_mean), ('Direct MLP', pred_direct),
                   ('Latent translation', pred_translated), ('Sensor self-reconstruction (uses target)', pred_self)]:
    result = metrics(pred[test], data['target'][test])
    print(name, 'RMSE=', round(result['rmse'], 4), 'per-axis MAE=', np.round(result['per_axis_mae'], 4))

seq = np.unique(data['sequence'][test])[0]
selected = data['sequence'] == seq
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for axis, ax in enumerate(axes):
    ax.plot(data['time'][selected], data['target'][selected, axis], 'k', label='Target')
    ax.plot(data['time'][selected], pred_direct[selected, axis], label='Direct MLP')
    ax.plot(data['time'][selected], pred_translated[selected, axis], label='Latent translation')
    ax.set_ylabel('xyz'[axis] + ' (nominal N)'); ax.grid(alpha=.2)
axes[0].legend(); axes[0].set_title(DOMAIN + ' | held-out episode')
axes[-1].set_xlabel('Synthetic sequence time (s)' if DOMAIN == 'synthetic' else 'Trigger-relative time (s)')
fig.tight_layout()
plt.show()


## 6. 看看 latent 空间

在传感器 latent 中，用圆点表示编码出的目标、叉号表示从仿真映射来的预测。两者使用同一个 PCA 投影；投影只根据训练集的传感器 latent 拟合。不能把两个空间各自降维后，凭图形相似就宣称已经对齐。

In [ ]:
with torch.no_grad():
    zr = sensor_ae.encoder(y).numpy()
    zm = translator.latent(x).numpy()
center = zr[train].mean(axis=0)
_, _, vt = np.linalg.svd(zr[train] - center, full_matrices=False)
basis = vt[:2].T
r = (zr[selected] - center) @ basis
m = (zm[selected] - center) @ basis
if r.shape[1] == 1:
    r = np.column_stack([r, np.zeros(len(r))])
    m = np.column_stack([m, np.zeros(len(m))])
color = np.linalg.norm(data['target'][selected], axis=1)
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(r[:, 0], r[:, 1], c=color, label='Sensor latent', cmap='viridis')
ax.scatter(m[:, 0], m[:, 1], c=color, marker='x', label='Mapped sim latent', cmap='viridis')
for a, b in zip(r[::3], m[::3]):
    ax.plot([a[0], b[0]], [a[1], b[1]], color='gray', alpha=.3)
ax.set(xlabel='Training PCA axis 1', ylabel='Training PCA axis 2', title=DOMAIN)
ax.legend(); fig.colorbar(sc, ax=ax, label='Target norm (nominal N)')
plt.show()


## 7. 练习与提交

1. 将 `WINDOW` 从 7 改成 1，重新运行。滞后响应为什么可能让单帧预测更难？
2. 将 `SENSOR_LATENT` 改为 2、4、8。比较自重建误差与跨模态误差；哪一部分限制了结果？
3. 将 `DOMAIN` 切换到 `synthetic`。哪些分数能解释教学生成器，哪些分数来自实测信号？
4. 比较直接 MLP 和 latent 映射。为什么中间层更多并不保证预测更准确？
5. 使用命令行参考实验中的 ridge 基线：如果线性模型已很好，复杂模型带来了什么？
6. 解释为什么 300 条合成序列仍只有 5 条原始运动来源，不能随机拆散帧来扩大测试集。

提交：一张三轴预测图、一张 latent 图、一张参数与误差表，以及对上面至少两个问题的解释。调整模型只看验证集；最终测试集用于一次独立报告。反复根据测试结果调参后，应把它视为开发集并另留测试数据。

编码转换不是改变真实物理，也不是把仿真力变成真机测量。这里学习的是两种观测表征之间的统计映射。